Lab 3 — Bronze → silver cleaning in PySpark (25 min, Free) · §3
Goal: turn orders_bronze (Lab 1) into two clean silver tables: one row per order, and one row per order item. Every step maps to a cleaning verb the exam names.
1. New notebook 03_silver, Python, serverless; paste the header lines.
2. Profile the mess first: display(spark.table(f"{CATALOG}.de_lab.orders_bronze").summary()) and count nulls per column. Note the string amount, mixed-case status, null customer_id and duplicates.
3. Run the orders cell. Expect 600 rows (order IDs 1–600).
4. Run the items cell, which uses explode on the nested items array and split on the SKU.
5. Compare dropDuplicates() with dropDuplicates(["order_id"]) on bronze: both give 600 here, but explain why the keyed version is safer when duplicates differ in one column.
6. Stretch: write the status cleanup in SQL with upper(coalesce(status, 'UNKNOWN')) and try_cast(amount AS DECIMAL(10,2)).

In [0]:
CATALOG = spark.sql("SELECT current_catalog()").first()[0]
BASE = f"/Volumes/{CATALOG}/de_lab/landing"

In [0]:
display(spark.table(f"{CATALOG}.de_lab.orders_bronze").summary())

In [0]:
from pyspark.sql import functions as F

bronze = spark.table(f"{CATALOG}.de_lab.orders_bronze")

orders_silver = (bronze
                 .dropDuplicates(["order_id"])  # dedup on the business key
                 .filter(F.col("order_id").isNotNull())
                 .withColumn("amount", F.col("amount").try_cast("decimal(10,2)"))   # standardize types
                 .withColumn("order_ts", F.to_timestamp("order_ts"))
                 .withColumn("order_date", F.to_date("order_ts"))
                 .withColumn("status", F.upper(F.coalesce(F.col("status"), F.lit("UNKNOWN"))))  # nulls + casing
                 .withColumnRenamed("coupon_code", "coupon")    # rename
                 .drop("items", "_rescued_data"))    # drop

orders_silver.write.mode("overwrite").saveAsTable(f"{CATALOG}.de_lab.orders_silver")

In [0]:
orders_silver = spark.table(f"{CATALOG}.de_lab.orders_silver")

orders_silver.count()

In [0]:
orders_silver.show()

In [0]:
items_silver = (bronze
                .dropDuplicates(["order_id"])
                .select("order_id", F.explode("items").alias("item"))   # one row per array element
                .select("order_id", 
                        F.col("item.sku").alias("sku"),
                        F.col("item.qty").cast("int").alias("qty"),
                        F.col("item.price").cast("decimal(10,2)").alias("price"))
                .withColumn("sku_number", F.split("sku", "-").getItem(1).cast("int"))) # split a column

items_silver.write.mode("overwrite").saveAsTable(f"{CATALOG}.de_lab.order_items_silver")

In [0]:
order_items_silver = spark.table(f"{CATALOG}.de_lab.order_items_silver")

order_items_silver.count()

In [0]:
order_items_silver.show()